# BigQuery Hands-On Lab — GCP Training Program
### Day 1 (Part 2) & Day 2: BigQuery Foundations, Advanced Features & Governance

This notebook walks through the full BigQuery agenda end-to-end against a
single running e-commerce dataset (`customers`, `products`, `orders`),
so every concept builds on the last instead of switching context.

**Run cells top to bottom, in order.** Each concept has a markdown
explanation cell immediately followed by a runnable code cell.

**Before you start:**
- You need a GCP project with billing enabled and the BigQuery API and
  Cloud Storage API enabled.
- Have `customers.csv`, `products.csv`, and `orders.json` ready to upload
  when prompted in Step 2 (from the training dataset package).
- Cost note: this lab uses a tiny dataset (under 200 total rows across all
  tables), so total cost across the full notebook should be a few cents at
  most — but BQML training and any GenAI/Vertex-connected steps are flagged
  individually below since those touch billed services beyond core BigQuery
  storage/query pricing.


## Configuration

Set your **project ID** and **GCS bucket name** here — every other cell in
this notebook references these variables, so you only configure once.

- `PROJECT_ID` — your GCP project ID (not the project *name* — check the
  GCP Console if unsure).
- `BUCKET_NAME` — a globally unique GCS bucket name. If it doesn't exist
  yet, this notebook creates it for you in Step 0.
- `DATASET_ID` — the BigQuery dataset name that will hold all lab tables.
- `LOCATION` — BigQuery/GCS region. `US` is a safe multi-region default.


In [ ]:
PROJECT_ID = "your-gcp-project-id"  #@param {type:"string"}
BUCKET_NAME = "your-gcs-bucket-name"  #@param {type:"string"}
DATASET_ID = "training_demo"  #@param {type:"string"}
LOCATION = "US"  #@param {type:"string"}

# Optional toggles for steps that need extra GCP setup or incur additional
# billed services (BQML training, Vertex AI embeddings for vector search,
# row access policies naming a real user). Leave off until you're ready.
RUN_BQML_STEP = False  #@param {type:"boolean"}
RUN_VECTOR_SEARCH_STEP = False  #@param {type:"boolean"}
RUN_RLS_STEP = False  #@param {type:"boolean"}
STUDENT_EMAIL = "student1@yourdomain.com"  #@param {type:"string"}

def TBL(name):
    """Fully-qualified, backtick-quoted table reference for f-string SQL."""
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   {BUCKET_NAME}")
print(f"Dataset:  {DATASET_ID}")
print(f"Location: {LOCATION}")


## Authenticate and set up clients

Colab needs to authenticate as you before it can touch your GCP project.
This opens a login prompt the first time you run it.


In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated.")


In [ ]:
!pip install --quiet google-cloud-bigquery google-cloud-storage

from google.cloud import bigquery
from google.cloud import storage
from google.cloud.exceptions import Conflict
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

def run_query(sql, label=None, expect_rows=True):
    """Run a query/DDL/DML statement and, if it returns rows, show them."""
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            df = job.result().to_dataframe()
            display(df)
        except Exception:
            job.result()
    else:
        job.result()
    if job.total_bytes_processed is not None:
        print(f"Bytes processed: {job.total_bytes_processed:,}")
    return job

print("BigQuery and Storage clients ready.")


## Step 0: Create the BigQuery dataset and GCS bucket

CLI equivalents (for reference — this cell does the same thing via the
Python SDK):
```
bq mk --dataset --location={LOCATION} {PROJECT_ID}:{DATASET_ID}
gsutil mb -l {LOCATION} gs://{BUCKET_NAME}/
```


In [ ]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
try:
    bq_client.create_dataset(dataset_ref)
    print(f"Created dataset {DATASET_ID}")
except Conflict:
    print(f"Dataset {DATASET_ID} already exists — continuing.")

bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"Created bucket {BUCKET_NAME}")
else:
    print(f"Bucket {BUCKET_NAME} already exists — continuing.")


## Step 1: Create the native tables

Three tables: `customers`, `products`, and `orders`. The `orders` table has
a nested/repeated column (`ARRAY<STRUCT<...>>`) for `line_items` — this is
what lets us demonstrate ARRAY/STRUCT/UNNEST later. It's also our
**partitioned + clustered** table (`PARTITION BY order_date`,
`CLUSTER BY region, status`) for the partitioning module in Step 7.


In [ ]:
run_query(f"""
CREATE OR REPLACE TABLE {TBL("customers")} (
  customer_id      INT64,
  first_name       STRING,
  last_name        STRING,
  email            STRING,
  city             STRING,
  country          STRING,
  region           STRING,
  customer_segment STRING,
  signup_date      DATE,
  churned          BOOL
)
""", label="Create customers table", expect_rows=False)

run_query(f"""
CREATE OR REPLACE TABLE {TBL("products")} (
  product_id   INT64,
  product_name STRING,
  category     STRING,
  subcategory  STRING,
  unit_price   FLOAT64,
  description  STRING
)
""", label="Create products table", expect_rows=False)

run_query(f"""
CREATE OR REPLACE TABLE {TBL("orders")} (
  order_id    INT64,
  customer_id INT64,
  region      STRING,
  order_date  DATE,
  status      STRING,
  line_items  ARRAY<STRUCT<
    product_id   INT64,
    product_name STRING,
    quantity     INT64,
    unit_price   FLOAT64
  >>
)
PARTITION BY order_date
CLUSTER BY region, status
""", label="Create orders table (partitioned + clustered)", expect_rows=False)


## Step 2: Upload the dataset files and load them into BigQuery

Run the cell below, then select `customers.csv`, `products.csv`, and
`orders.json` all at once from the file picker.


In [ ]:
from google.colab import files
print("Upload customers.csv, products.csv, and orders.json (select all three).")
uploaded = files.upload()


Now push each file to the GCS bucket, then load it into its matching
BigQuery table. `WRITE_TRUNCATE` makes this safe to re-run — each run
replaces the table contents rather than appending duplicates.


In [ ]:
def upload_to_gcs(local_filename):
    blob = bucket.blob(f"{DATASET_ID}/{local_filename}")
    blob.upload_from_filename(local_filename)
    uri = f"gs://{BUCKET_NAME}/{DATASET_ID}/{local_filename}"
    print(f"Uploaded {local_filename} -> {uri}")
    return uri

customers_uri = upload_to_gcs("customers.csv")
products_uri  = upload_to_gcs("products.csv")
orders_uri    = upload_to_gcs("orders.json")


In [ ]:
# Load customers.csv (CSV -> existing schema, header row skipped)
csv_job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
bq_client.load_table_from_uri(customers_uri, TBL("customers"), job_config=csv_job_config).result()
print("customers loaded:", bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.customers").num_rows, "rows")

bq_client.load_table_from_uri(products_uri, TBL("products"), job_config=csv_job_config).result()
print("products loaded:", bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.products").num_rows, "rows")

# Load orders.json (nested JSON -> existing schema with ARRAY<STRUCT<...>>)
json_job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
bq_client.load_table_from_uri(orders_uri, TBL("orders"), job_config=json_job_config).result()
print("orders loaded:", bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.orders").num_rows, "rows")


## Step 3: Simple SQL queries

**3a — Basic SELECT with LIMIT.** Teaching point: always `LIMIT` on a first
look at a new table — a habit worth building in students so they never
blindly scan a huge table.


In [ ]:
run_query(f"""
SELECT customer_id, first_name, last_name, region, customer_segment
FROM {TBL("customers")}
LIMIT 10
""", label="3a. Basic SELECT")


**3b — Filtering + sorting.** Reinforces `WHERE` + `ORDER BY` evaluation
order from the fundamentals lessons, now against real rows.


In [ ]:
run_query(f"""
SELECT product_name, category, unit_price
FROM {TBL("products")}
WHERE category = 'Electronics' AND unit_price > 2000
ORDER BY unit_price DESC
""", label="3b. Filter + sort")


**3c — Aggregation with `COUNTIF`.** Teaching point: `COUNTIF(condition)` is
a BigQuery convenience over `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` —
a small "BigQuery makes this easier" moment worth calling out.


In [ ]:
run_query(f"""
SELECT region, COUNT(*) AS num_customers, COUNTIF(churned) AS churned_count
FROM {TBL("customers")}
GROUP BY region
ORDER BY num_customers DESC
""", label="3c. Aggregation")


## Step 4: Table joins (including UNNEST on nested `line_items`)

**4a — Standard INNER JOIN** — orders to customers.


In [ ]:
run_query(f"""
SELECT o.order_id, o.order_date, o.status, c.first_name, c.last_name, c.region
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC
LIMIT 15
""", label="4a. INNER JOIN")


**4b — Unpack nested `line_items`, then join to `products`** for full order
detail. Teaching point: `UNNEST` behaves like an implicit join against the
array, turning nested rows into flat ones you can then join further. This is
the bridge between the "joins" lesson and the "ARRAY/STRUCT" lesson.


In [ ]:
run_query(f"""
SELECT
  o.order_id,
  o.order_date,
  c.first_name,
  item.product_name,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
ORDER BY o.order_date DESC
LIMIT 15
""", label="4b. UNNEST + JOIN")


**4c — Full revenue rollup** (orders -> line_items -> products -> customers).
Teaching point: a good "capstone" query for the joins section — touches
every table, nested data, filtering, and aggregation at once.


In [ ]:
run_query(f"""
SELECT
  c.region,
  p.category,
  SUM(item.quantity * item.unit_price) AS total_revenue
FROM {TBL("orders")} o,
UNNEST(o.line_items) AS item
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
JOIN {TBL("products")} p ON item.product_id = p.product_id
WHERE o.status = 'Completed'
GROUP BY c.region, p.category
ORDER BY total_revenue DESC
""", label="4c. Full revenue rollup")


## Step 5: Views

A view is a saved query — it stores no data; it re-runs the underlying SQL
against live data every time it's queried.

**Teaching points:**
- No storage cost — a view holds zero bytes; billing is based on bytes
  scanned from the underlying tables at query time.
- Always fresh — since it re-runs live, it never goes stale.
- Simplifies complex logic — analysts querying the view don't need to know
  about the join or the UNNEST underneath.
- Downside — every query against the view redoes the full join/unnest work.
  If many dashboards hit this constantly, that's repeated compute cost —
  the natural segue into materialized views (Step 6).


In [ ]:
run_query(f"""
CREATE OR REPLACE VIEW {TBL("v_completed_order_details")} AS
SELECT
  o.order_id,
  o.order_date,
  o.region,
  c.customer_segment,
  item.product_name,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
""", label="Create view", expect_rows=False)

run_query(f"""
SELECT region, SUM(line_total) AS revenue
FROM {TBL("v_completed_order_details")}
GROUP BY region
""", label="Query the view")


## Step 6: Materialized Views

Unlike a view, a materialized view actually **stores** the precomputed
result and incrementally refreshes it as base tables change — trading
storage cost for query speed and lower repeated compute.

**View vs Materialized View:**

| | View | Materialized View |
|---|---|---|
| Stores data? | No | Yes (precomputed result) |
| Freshness | Always live | Auto-refreshed incrementally |
| Query cost | Full cost each run | Cheap — reads precomputed data |
| Storage cost | None | Yes — pay for the stored result |
| Best for | Hiding complexity, low-frequency queries | Dashboards, repeated aggregations |

**Caveat to flag in class:** materialized views have restrictions on
supported SQL (limited join patterns, a defined set of supported aggregate
functions — `SUM`, `COUNT`, `AVG`, `MIN`, `MAX`, `COUNT(DISTINCT)` in some
cases). Students who try to build one on an arbitrary multi-join or
window-function query may hit an error — tell them to check current docs,
since this surface evolves.


In [ ]:
run_query(f"""
CREATE MATERIALIZED VIEW {TBL("mv_region_daily_revenue")} AS
SELECT
  o.region,
  o.order_date,
  SUM(item.quantity * item.unit_price) AS daily_revenue,
  COUNT(DISTINCT o.order_id) AS num_orders
FROM {TBL("orders")} o,
UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
GROUP BY o.region, o.order_date
""", label="Create materialized view", expect_rows=False)

run_query(f"""
SELECT * FROM {TBL("mv_region_daily_revenue")}
WHERE region = 'APAC'
ORDER BY order_date DESC
""", label="Query the materialized view")


## Step 7: Partitioning — and proving the advantage

The `orders` table is already `PARTITION BY order_date, CLUSTER BY region,
status` (see Step 1). Here we demonstrate *why* that matters, by comparing
against an unpartitioned copy.

**7a** — create the unpartitioned copy for comparison.


In [ ]:
run_query(f"""
CREATE OR REPLACE TABLE {TBL("orders_unpartitioned")} AS
SELECT * FROM {TBL("orders")}
""", label="Create unpartitioned copy", expect_rows=False)


**7b** — run the same filtered query against both, and compare bytes
processed (printed after each query). This is the number to put on screen
side-by-side in class.


In [ ]:
print("Partitioned table:")
run_query(f"""
SELECT * FROM {TBL("orders")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'
""", label="Filtered query - partitioned")

print("\nUnpartitioned copy:")
run_query(f"""
SELECT * FROM {TBL("orders_unpartitioned")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'
""", label="Filtered query - unpartitioned")


Teaching point: the partitioned version processes less data, because
BigQuery uses **partition pruning** — it skips reading file blocks for
dates outside the range entirely, rather than scanning everything and
filtering afterward. At only ~70 rows this difference looks small in
absolute terms — tell students explicitly: *"at this toy scale the savings
look trivial, but this exact pruning mechanism is what turns a 500 GB scan
into a 2 GB scan on a real production table — the behavior, not the byte
count, is the lesson."*

**7c** — show clustering's advantage stacked on top of partitioning.


In [ ]:
run_query(f"""
SELECT * FROM {TBL("orders")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'
  AND region = 'APAC'
  AND status = 'Completed'
""", label="Partition + cluster pruning")


Since `orders` is `CLUSTER BY region, status`, within the already-pruned
date partition, BigQuery also sorts/organizes data by region and status —
so this filter skips further irrelevant blocks *within* the partition.
Partitioning narrows down to the right time range; clustering narrows down
further within that range.

**Summary — partitioning advantages:**
- Cost reduction: billed by bytes scanned; pruning skips irrelevant
  partitions entirely.
- Speed: less data read = faster queries.
- Manageability: easy to expire old partitions automatically (partition
  expiration setting), e.g. auto-delete data older than 2 years.
- Clustering stacks on top: further pruning on non-date columns, at no
  extra storage cost — BigQuery only allows **one** partition column, so
  clustering is how you get additional pruning dimensions.


## Step 8 (Day 1 Part 2): External Table / BigLake

Query the same `orders.json` directly from GCS — no load step, no bytes
copied into BigQuery managed storage.


In [ ]:
run_query(f"""
CREATE OR REPLACE EXTERNAL TABLE {TBL("orders_external")}
OPTIONS (
  format = 'NEWLINE_DELIMITED_JSON',
  uris = ['{orders_uri}']
)
""", label="Create external table", expect_rows=False)

run_query(f"""
SELECT order_id, order_date, status
FROM {TBL("orders_external")}
LIMIT 10
""", label="Query the external table")


Teaching point: `orders_external` is queried exactly like a native table,
but the underlying bytes stay in GCS. This is the foundation for BigLake —
governed, uniform access control over data that lives outside BigQuery
storage (GCS, and by extension Iceberg/Parquet tables).


## Step 9 (Day 2): BigQuery ML — predict churn

Train a logistic regression model directly in SQL to predict `churned`
from customer segment, region, order count, and lifetime value.

**Cost note:** BQML training is a billed operation beyond simple
query/storage pricing. This toggle defaults to off — set
`RUN_BQML_STEP = True` in the configuration cell above before running.


In [ ]:
if RUN_BQML_STEP:
    run_query(f"""
    CREATE OR REPLACE MODEL {TBL("churn_model")}
    OPTIONS(model_type='logistic_reg', input_label_cols=['churned']) AS
    SELECT
      c.customer_segment,
      c.region,
      COUNT(o.order_id) AS num_orders,
      IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value,
      c.churned
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.churned, c.customer_id
    """, label="Train churn_model", expect_rows=False)

    run_query(f"""
    SELECT * FROM ML.EVALUATE(MODEL {TBL("churn_model")})
    """, label="Evaluate churn_model")
else:
    print("RUN_BQML_STEP is False — set it to True above to train the model live.")


## Step 10 (Day 2): Time travel / change history

Mutate a row, then query the table's state *before* the change using
`FOR SYSTEM_TIME AS OF`.


In [ ]:
# Snapshot the current status first.
before_df = run_query(f"""
SELECT order_id, status FROM {TBL("orders")} WHERE order_id = 1001
""", label="Status before update")

run_query(f"""
UPDATE {TBL("orders")} SET status = 'Cancelled' WHERE order_id = 1001
""", label="Update order 1001", expect_rows=False)

run_query(f"""
SELECT order_id, status FROM {TBL("orders")} WHERE order_id = 1001
""", label="Status after update")


In [ ]:
# Time travel — query the table as it looked a few minutes ago, before the
# UPDATE above. Adjust the INTERVAL if you run this well after the update.
run_query(f"""
SELECT order_id, status
FROM {TBL("orders")}
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 5 MINUTE)
WHERE order_id = 1001
""", label="Time travel query")


## Step 11 (Day 2): Vector Search for RAG

This step needs a **remote connection to a Vertex AI embedding model**,
which must be created once per project outside this notebook:

```
bq mk --connection --connection_type=CLOUD_RESOURCE \
  --location={LOCATION} embedding_conn
```

Then grant the connection's service account the Vertex AI User role in
IAM. Once that's done, set `RUN_VECTOR_SEARCH_STEP = True` above.

**Honest caveat:** BigQuery's vector index behavior at very small data
scales (like our 26 products) may fall back to brute-force search rather
than using the index — worth telling students index behavior is tuned for
much larger tables, and the exact thresholds are evolving as the feature
matures.


In [ ]:
if RUN_VECTOR_SEARCH_STEP:
    CONNECTION_ID = f"{PROJECT_ID}.{LOCATION}.embedding_conn"  # adjust if named differently

    run_query(f"""
    CREATE OR REPLACE MODEL {TBL("embedding_model")}
    REMOTE WITH CONNECTION `{CONNECTION_ID}`
    OPTIONS (endpoint = 'text-embedding-004')
    """, label="Create embedding model", expect_rows=False)

    run_query(f"""
    CREATE OR REPLACE TABLE {TBL("product_embeddings")} AS
    SELECT product_id, product_name, ml_generate_embedding_result AS embedding
    FROM ML.GENERATE_EMBEDDING(
      MODEL {TBL("embedding_model")},
      (SELECT product_id, product_name, description AS content FROM {TBL("products")})
    )
    """, label="Generate product embeddings", expect_rows=False)

    run_query(f"""
    CREATE VECTOR INDEX product_index
    ON {TBL("product_embeddings")}(embedding)
    OPTIONS (index_type = 'IVF', distance_type = 'COSINE')
    """, label="Create vector index", expect_rows=False)

    run_query(f"""
    SELECT base.product_name, distance
    FROM VECTOR_SEARCH(
      TABLE {TBL("product_embeddings")}, 'embedding',
      (SELECT embedding FROM {TBL("product_embeddings")} WHERE product_name = 'Wireless Mouse'),
      top_k => 5
    )
    """, label="Vector search - similar products")
else:
    print("RUN_VECTOR_SEARCH_STEP is False — requires a Vertex AI remote connection. See markdown above.")


## Step 12 (Day 2): Row-Level Security

Restrict a real IAM principal to only see `APAC` rows in `orders`. Set
`STUDENT_EMAIL` above to a real user/group in your domain, and
`RUN_RLS_STEP = True`, before running — row access policies grant to real
IAM identities, so this isn't meaningful against a placeholder email.


In [ ]:
if RUN_RLS_STEP:
    run_query(f"""
    CREATE ROW ACCESS POLICY apac_only
    ON {TBL("orders")}
    GRANT TO ('user:{STUDENT_EMAIL}')
    FILTER USING (region = 'APAC')
    """, label="Create row access policy", expect_rows=False)
    print("Policy created. Query orders as that user to see the filter apply.")
else:
    print("RUN_RLS_STEP is False — set STUDENT_EMAIL and the toggle above to run this live.")


## Step 13 (Day 2): Column-level encryption (AEAD)

Encrypt the `email` column at query time using a session keyset. This is
cheap to run and doesn't need extra GCP setup — safe to demo live as-is.


In [ ]:
run_query(f"""
DECLARE keyset BYTES DEFAULT KEYS.NEW_KEYSET('AEAD_AES_GCM_256');

SELECT customer_id,
       AEAD.ENCRYPT(keyset, email, CAST(customer_id AS STRING)) AS encrypted_email
FROM {TBL("customers")}
LIMIT 10
""", label="AEAD column encryption")


Teaching point: the keyset here is generated inline and discarded when the
query ends — in production you'd store the keyset in Cloud KMS and
reference it consistently so encrypted values can actually be decrypted
later. This demo shows the encryption mechanics, not full key-management
practice.


## Step 14 (Day 2): Remote functions

Remote functions call an external service (e.g., a Cloud Function or Cloud
Run endpoint) from inside a SQL query. This requires a deployed endpoint
and a remote connection, both outside the scope of this lab dataset — shown
here as reference syntax only, not executed.


In [ ]:
reference_sql = f"""
CREATE OR REPLACE FUNCTION {TBL("convert_to_usd")}(amount FLOAT64, currency STRING)
RETURNS FLOAT64
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.remote_conn`
OPTIONS (endpoint = 'https://YOUR_CLOUD_FUNCTION_URL')
"""
print("Reference syntax only (not executed) -- deploy an endpoint first:\n")
print(reference_sql)


## Wrap-up

That's the full BigQuery agenda against one running dataset. Natural next
steps for your Day 2 close: Data Lineage (Dataplex/BigQuery lineage API) is
a governance-layer feature best shown in the Cloud Console UI rather than
via SQL — worth a live console walkthrough rather than another notebook
cell.
